In [71]:
import sqlite3
import pandas as pd
from datetime import datetime
pd.set_option('display.width', 1000)
conn = sqlite3.connect('../data/checking-logs.sqlite') 

In [72]:
conn.execute("DROP TABLE IF EXISTS datamart")
conn.commit()
create_datamart_query = """
CREATE TABLE datamart AS
SELECT 
    c.uid,
    c.labname,
    c.timestamp AS first_commit_ts,
    MIN(p.datetime) AS first_view_ts
FROM 
    checker c
LEFT JOIN 
    pageviews p ON c.uid = p.uid
WHERE 
    c.status = 'ready'
    AND c.numTrials = 1
    AND c.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
    AND c.uid LIKE 'user_%'
GROUP BY 
    c.uid, c.labname, c.timestamp
"""
try:
    conn.execute(create_datamart_query)
    conn.commit()
    print("Таблица datamart успешно создана")
except Exception as e:
    print(f"Ошибка при создании таблицы datamart: {e}")
    conn.close()
    exit()



Таблица datamart успешно создана


In [73]:
try:
    datamart = pd.io.sql.read_sql(
        "SELECT * FROM datamart", 
        conn, 
        parse_dates=['first_commit_ts', 'first_view_ts']
    )
    print("Данные успешно загружены из таблицы datamart")
except Exception as e:
    print(f"Ошибка при чтении данных: {e}")
    conn.close()
    exit()


Данные успешно загружены из таблицы datamart


In [74]:
test = datamart[datamart['first_view_ts'].notna()].copy()
control = datamart[datamart['first_view_ts'].isna()].copy()

In [75]:
avg_view_ts = test['first_view_ts'].mean()
control['first_view_ts'] = control['first_view_ts'].fillna(avg_view_ts)

In [76]:
try:
    conn.execute("DROP TABLE IF EXISTS test")
    conn.execute("DROP TABLE IF EXISTS control")
    conn.commit()
    
    test.to_sql('test', conn, if_exists='fail', index=False)
    control.to_sql('control', conn, if_exists='fail', index=False)
    print("Таблицы test и control успешно сохранены")
except Exception as e:
    print(f"Ошибка при сохранении таблиц: {e}")
finally:
   
    conn.close()
    print("Соединение с базой данных закрыто")

Таблицы test и control успешно сохранены
Соединение с базой данных закрыто
